# Lens Notebook

This notebook shows how to use both `LogitLens` and `TunedLens` with `ModelWithSplitPoints` across several model families.

Origins:
- Logit Lens: nostalgebraist, *Interpreting GPT: the logit lens*
- Related vocabulary-space analysis: Geva et al. (2022)
- Tuned Lens: Belrose et al. (2023)

The tuned lens implementation supports three initialization modes:
- `logit_lens`: identity initialization so tuning starts from the plain logit-lens behavior
- `xavier`: Xavier uniform initialization with zero bias
- `default`: the current `torch.nn.Linear` initialization

The notebook uses tiny checkpoints so it stays light enough for quick experimentation.
Some of them are random checkpoints, so semantic quality is not the goal here: the examples are mainly meant to illustrate the API and the decodability metrics.

Metric interpretation:
- `mean_target_probability`: higher is better
- `target_cross_entropy`: lower is better
- `perplexity`: lower is better for causal language models
- `kl_divergence_to_model`: lower is better and differentiable, which makes it useful as a regularization target for linear decodability
- `model_top1_agreement`: agreement with the final model argmax

The raw `explain()` and `lens()` outputs are tensor-first and use `top_indices` / `top_scores`.
Human-readable decoding is handled by the visualization layer.


In [17]:
from transformers import AutoModelForCausalLM, AutoModelForMaskedLM, AutoModelForSequenceClassification, AutoTokenizer

from interpreto import LogitLens, ModelWithSplitPoints, TunedLens


def summarize_metrics(metrics, split_point):
    keys = [
        'target_source',
        'nb_evaluated_elements',
        'mean_target_probability',
        'target_cross_entropy',
        'target_accuracy',
        'mean_max_probability',
        'kl_divergence_to_model',
        'model_top1_agreement',
        'perplexity',
    ]
    return {key: metrics[split_point][key] for key in keys if key in metrics[split_point]}


## Causal Language Model

We start with a small GPT-style model and inspect two prompts at once.


In [18]:
causal_model_name = 'hf-internal-testing/tiny-random-gpt2'
causal_model = AutoModelForCausalLM.from_pretrained(causal_model_name)
causal_tokenizer = AutoTokenizer.from_pretrained(causal_model_name)
if causal_tokenizer.pad_token is None:
    causal_tokenizer.pad_token = causal_tokenizer.eos_token

causal_model_with_split_points = ModelWithSplitPoints(
    causal_model,
    tokenizer=causal_tokenizer,
    split_points='transformer.h.1.mlp',
    batch_size=2,
    device_map='cpu',
)

causal_examples = [
    'Interpreto is useful.',
    'Interpreto helps explain models.',
]

causal_model_with_split_points.split_points


['transformer.h.1.mlp']

In [19]:
causal_logit_lens = LogitLens(causal_model_with_split_points, top_k=3)
causal_logit_explanations = causal_logit_lens.lens(causal_examples)


In [20]:
causal_logit_explanations['transformer.h.1.mlp']['top_indices'][0, 0], causal_logit_explanations['transformer.h.1.mlp']['top_scores'][0, 0]


KeyError: 'top_indices'

In [ ]:
causal_logit_metrics = causal_logit_lens.metrics(causal_examples)
summarize_metrics(causal_logit_metrics, 'transformer.h.1.mlp')


{'target_source': 'next_token',
 'nb_evaluated_elements': 29,
 'mean_target_probability': 0.00100420240778476,
 'target_cross_entropy': 6.909729480743408,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.0014203732134774327,
 'kl_divergence_to_model': 0.008143223822116852,
 'model_top1_agreement': 0.03448275849223137,
 'perplexity': 1001.9761352539062}

## Masked Language Model

The same `LogitLens` workflow also works on masked-language-model checkpoints.


In [ ]:
masked_model_name = 'hf-internal-testing/tiny-random-bert'
masked_model = AutoModelForMaskedLM.from_pretrained(masked_model_name)
masked_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

masked_model_with_split_points = ModelWithSplitPoints(
    masked_model,
    tokenizer=masked_tokenizer,
    split_points='bert.encoder.layer.1.output',
    batch_size=2,
    device_map='cpu',
)

masked_examples = [
    'Interpreto is useful',
    'Interpreto explains transformers',
]

masked_model_with_split_points.split_points


BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


/tmp/interpreto-lens-venv/lib64/python3.11/site-packages/nnsight/intervention/envoy.py:823: UserWarning: Module `model.bert.encoder.layer.0.attention` of type `<class 'transformers.models.bert.modeling_bert.BertAttention'>` has pre-defined a `output` attribute. nnsight access for `output` will be mounted at `.nns_output` instead of `.output` for this module only.
  warnings.warn(
/tmp/interpreto-lens-venv/lib64/python3.11/site-packages/nnsight/intervention/envoy.py:823: UserWarning: Module `model.bert.encoder.layer.0` of type `<class 'transformers.models.bert.modeling_bert.BertLayer'>` has pre-defined a `output` attribute. nnsight access for `output` will be mounted at `.nns_output` instead of `.output` for this module only.
  warnings.warn(
/tmp/interpreto-lens-venv/lib64/python3.11/site-packages/nnsight/intervention/envoy.py:823: UserWarning: Module `model.bert.encoder.layer.1.attention` of type `<class 'transformers.models.bert.modeling_bert.BertAttention'>` has pre-defined a `outpu

['bert.encoder.layer.1.output']

In [ ]:
masked_logit_lens = LogitLens(masked_model_with_split_points, top_k=4)
masked_logit_explanations = masked_logit_lens.lens(masked_examples)


In [ ]:
masked_logit_metrics = masked_logit_lens.metrics(masked_examples)
summarize_metrics(masked_logit_metrics, 'bert.encoder.layer.1.output')


{'target_source': 'token_identity',
 'nb_evaluated_elements': 48,
 'mean_target_probability': 0.0009017205447889864,
 'target_cross_entropy': 7.0173869132995605,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.0012729082955047488,
 'kl_divergence_to_model': 2.1701562218368053e-06,
 'model_top1_agreement': 0.9583333134651184}

## Sequence Classification

For classification checkpoints, the same framework exposes intermediate label distributions and classification-oriented scores.


In [ ]:
classification_model = AutoModelForSequenceClassification.from_pretrained(masked_model_name)
classification_tokenizer = AutoTokenizer.from_pretrained(masked_model_name)

classification_model_with_split_points = ModelWithSplitPoints(
    classification_model,
    tokenizer=classification_tokenizer,
    split_points='bert.encoder.layer.1.output',
    batch_size=2,
    device_map='cpu',
)

classification_examples = [
    'Interpreto is helpful',
    'Interpreto is practical',
]
classification_targets = [1, 0]

classification_model_with_split_points.split_points


['bert.encoder.layer.1.output']

In [ ]:
classification_logit_lens = LogitLens(classification_model_with_split_points, top_k=2)
classification_logit_explanations = classification_logit_lens.lens(classification_examples)


In [ ]:
classification_logit_metrics = classification_logit_lens.metrics(
    classification_examples,
    targets=classification_targets,
)
summarize_metrics(classification_logit_metrics, 'bert.encoder.layer.1.output')


{'target_source': 'provided_targets',
 'nb_evaluated_elements': 2,
 'mean_target_probability': 0.4999966025352478,
 'target_cross_entropy': 0.6932074427604675,
 'target_accuracy': 0.5,
 'mean_max_probability': 0.5051708817481995,
 'kl_divergence_to_model': 5.960464477539063e-08,
 'model_top1_agreement': 1.0}

## Tuned Lens On A Small Dataset

The final section fits a `TunedLens` on a tiny text collection for the causal model.
This is only a small demonstration, but it shows how the decodability metrics can be tracked before and after tuning.


In [ ]:
supported_modes = ['logit_lens', 'xavier', 'default']
[TunedLens(causal_model_with_split_points, top_k=3, initialization_mode=mode).initialization_mode for mode in supported_modes]


['logit_lens', 'xavier', 'default']

In [ ]:
tuning_texts = [
    'Interpreto is useful.',
    'Interpreto helps explain transformers.',
    'Interpreto makes debugging easier.',
    'Interpreto is practical for analysis.',
]

tuned_lens = TunedLens(causal_model_with_split_points, top_k=3, initialization_mode='logit_lens')
pre_tuning_metrics = summarize_metrics(tuned_lens.metrics(causal_examples), 'transformer.h.1.mlp')
history = tuned_lens.fit(tuning_texts, epochs=2, batch_size=2)
post_tuning_metrics = summarize_metrics(tuned_lens.metrics(causal_examples), 'transformer.h.1.mlp')
history


{'loss': [0.008252160623669624, 0.00788214709609747],
 'split_points': ['transformer.h.1.mlp'],
 'epochs': 2}

In [ ]:
{'before': pre_tuning_metrics, 'after': post_tuning_metrics}


{'before': {'target_source': 'next_token',
  'nb_evaluated_elements': 29,
  'mean_target_probability': 0.00100420240778476,
  'target_cross_entropy': 6.909729480743408,
  'target_accuracy': 0.0,
  'mean_max_probability': 0.0014203732134774327,
  'kl_divergence_to_model': 0.008143223822116852,
  'model_top1_agreement': 0.03448275849223137,
  'perplexity': 1001.9761352539062},
 'after': {'target_source': 'next_token',
  'nb_evaluated_elements': 29,
  'mean_target_probability': 0.001015501911751926,
  'target_cross_entropy': 6.8977885246276855,
  'target_accuracy': 0.0,
  'mean_max_probability': 0.0014187912456691265,
  'kl_divergence_to_model': 0.007444031070917845,
  'model_top1_agreement': 0.03448275849223137,
  'perplexity': 990.082763671875}}

In [ ]:
tuned_lens_explanations = tuned_lens.lens([
    'Interpreto helps debug transformers.',
    'Interpreto makes analysis practical.',
])


In [ ]:
held_out_metrics = tuned_lens.metrics([
    'Interpreto helps debug transformers.',
    'Interpreto makes analysis practical.',
])
summarize_metrics(held_out_metrics, 'transformer.h.1.mlp')


{'target_source': 'next_token',
 'nb_evaluated_elements': 36,
 'mean_target_probability': 0.001018113223835826,
 'target_cross_entropy': 6.89621639251709,
 'target_accuracy': 0.0,
 'mean_max_probability': 0.0014180490979924798,
 'kl_divergence_to_model': 0.007444901391863823,
 'model_top1_agreement': 0.0,
 'perplexity': 988.5274047851562}